# 🎨 ToneFit ML
## Machine Learning-Based Personal Color Season Prediction
### Polytechnic University of the Philippines — Data Science Pilot Study

---

**Personal Color Seasons:**
| Season | Undertone | Features |
|--------|-----------|----------|
| 🌸 Spring | Warm | Light, clear, peachy |
| ☁️ Summer | Cool | Light, muted, ashy |
| 🍂 Autumn | Warm | Deep, muted, earthy |
| ❄️ Winter | Cool | Deep, clear, high contrast |

**Pipeline:**
1. Setup & Mount Drive
2. Clone GitHub Repo
3. Install Dependencies
4. Collect Dataset
5. Preprocessing & Feature Extraction
6. Exploratory Data Analysis (EDA)
7. Traditional ML Training (SVM + Random Forest)
8. Deep Learning Training (MobileNetV2)
9. Model Evaluation & Comparison
10. Prediction Demo

---
## ⚙️ STEP 0 — Setup
### Mount Google Drive + Clone Repo

> **Run this first every time you open this notebook.**  
> Your dataset and models are saved to Google Drive so they don't disappear when Colab resets.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create project folder in Drive
import os
PROJECT_DIR = '/content/drive/MyDrive/ToneFit'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'✅ Project folder ready: {PROJECT_DIR}')

In [ ]:
# Clone GitHub repo
import os

REPO_DIR = '/content/ToneFit'

if os.path.exists(REPO_DIR):
    print('Repo already exists, pulling latest...')
    os.chdir(REPO_DIR)
    !git pull
else:
    !git clone https://github.com/ajipal/ToneFit.git
    os.chdir(REPO_DIR)

print(f'\n✅ Working directory: {os.getcwd()}')
print('\n📁 Files in repo:')
!ls -la

In [ ]:
# Install all dependencies
!pip install icrawler opencv-python-headless scikit-learn tensorflow \
             pandas numpy matplotlib seaborn scikit-image imagehash \
             colormath tqdm -q

print('✅ All dependencies installed')

---
## 📥 STEP 1 — Data Collection

> Downloads face images of verified celebrities per season using Google Images.  
> Labels come from publicly documented professional color diagnoses.
>
> ⏱️ Takes about 10-15 minutes per season.  
> 💡 Split among your group — each person runs one season on their own laptop, then upload to the shared Drive folder.

**To run only one season, change `RUN_SEASONS` below.**

In [ ]:
import os

# Set dataset path to Google Drive so it persists
DATASET_DIR = '/content/drive/MyDrive/ToneFit/dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

# Check what we already have
seasons = ['spring', 'summer', 'autumn', 'winter']
print('📊 Current dataset status:')
total = 0
for s in seasons:
    path = os.path.join(DATASET_DIR, s)
    os.makedirs(path, exist_ok=True)
    count = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    total += count
    bar = '█' * (count // 3)
    print(f'  {s:<8}: {count:>4} images  {bar}')
print(f'\n  TOTAL   : {total} images')

In [ ]:
# ─── CONFIGURATION ────────────────────────────────────────────────────────────
# Change this to run only specific seasons
# None = run all seasons
# ["spring"] = run only spring
RUN_SEASONS = None  # Change to ["spring"], ["summer"], etc.

# ─── CELEBRITY LIST ───────────────────────────────────────────────────────────
CELEBRITIES = {
    "spring": [
        # Filipino
        "Anne Curtis actress Philippines face",
        "Julia Barretto actress Philippines face",
        "Marian Rivera actress Philippines face",
        "James Reid actor Philippines face",
        # Korean
        "IU Korean singer face",
        "Yoona SNSD Korean singer face",
        "Kim Chaewon Le Sserafim kpop face",
        "Kim Soo-hyun Korean actor face",
        "Jung Hae-in Korean actor face",
        "Felix Stray Kids kpop face",
        # Western
        "Chris Hemsworth actor face",
        "Pedro Pascal actor face",
        "Sterling K Brown actor face",
        "Emma Stone actress face",
        "Ariana Grande singer face",
    ],
    "summer": [
        # Filipino
        "Jodi Sta Maria actress Philippines face",
        "Janine Gutierrez actress Philippines face",
        "Shaina Magdayao actress Philippines face",
        "Richard Gutierrez actor Philippines face",
        "Matteo Guidicelli actor Philippines face",
        # Korean
        "Son Ye-jin Korean actress face",
        "Irene Red Velvet kpop face",
        "Jang Wonyoung IVE kpop face",
        "Taeyeon SNSD Korean singer face",
        "Cha Eunwoo ASTRO kpop face",
        "Jimin BTS kpop face",
        # Western
        "Taylor Swift singer face",
        "Nicole Kidman actress face",
        "Robert Pattinson actor face",
        "Timothee Chalamet actor face",
    ],
    "autumn": [
        # Filipino
        "Kathryn Bernardo actress Philippines face",
        "Nadine Lustre actress Philippines face",
        "Gabbi Garcia actress Philippines face",
        "Coleen Garcia actress Philippines face",
        "Liza Soberano actress Philippines face",
        "Piolo Pascual actor Philippines face",
        "Daniel Padilla actor Philippines face",
        "Enrique Gil actor Philippines face",
        "Coco Martin actor Philippines face",
        "Joshua Garcia actor Philippines face",
        # Korean
        "Jennie Blackpink kpop face",
        "Jaehyun NCT kpop face",
        "V BTS Taehyung face",
        # Western
        "Beyonce singer face",
        "Oscar Isaac actor face",
    ],
    "winter": [
        # Filipino
        "Heart Evangelista actress Philippines face",
        "Pia Wurtzbach Philippines face",
        "Alden Richards actor Philippines face",
        "Dingdong Dantes actor Philippines face",
        "Paulo Avelino actor Philippines face",
        "Xian Lim actor Philippines face",
        "Donny Pangilinan actor Philippines face",
        # Korean
        "Jisoo Blackpink kpop face",
        "Suga BTS Yoongi face",
        "Jin BTS face",
        "Hyun Bin Korean actor face",
        "Song Hye-kyo Korean actress face",
        # Western
        "Megan Fox actress face",
        "Dua Lipa singer face",
        "Keanu Reeves actor face",
    ],
}

print('✅ Celebrity list loaded')
for s, celebs in CELEBRITIES.items():
    print(f'  {s:<8}: {len(celebs)} celebrities → target ~{len(celebs)*5} images')

In [ ]:
import cv2
import shutil
import logging
from icrawler.builtin import GoogleImageCrawler

CASCADE_PATH = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(CASCADE_PATH)
RAW_DIR      = '/content/drive/MyDrive/ToneFit/raw'
REJECTED_DIR = '/content/drive/MyDrive/ToneFit/rejected'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(REJECTED_DIR, exist_ok=True)

def download_celebrity(query, save_dir, max_num=5):
    try:
        crawler = GoogleImageCrawler(
            storage={'root_dir': save_dir},
            log_level=logging.WARNING,
            feeder_threads=1, parser_threads=1, downloader_threads=2
        )
        crawler.crawl(keyword=query, max_num=max_num, filters={'type': 'face'})
    except Exception as e:
        print(f'    ⚠️ Error for {query}: {e}')

def crop_face(img_path, output_path, min_size=60):
    img = cv2.imread(img_path)
    if img is None: return False
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(min_size, min_size))
    if len(faces) == 0: return False
    x, y, w, h = max(faces, key=lambda f: f[2]*f[3])
    pad = int(0.20 * max(w, h))
    x1, y1 = max(0, x-pad), max(0, y-pad)
    x2, y2 = min(img.shape[1], x+w+pad), min(img.shape[0], y+h+pad)
    face_crop = cv2.resize(img[y1:y2, x1:x2], (224, 224))
    cv2.imwrite(output_path, face_crop)
    return True

def collect_season(season, celebrities):
    raw_dir    = os.path.join(RAW_DIR, season)
    season_dir = os.path.join(DATASET_DIR, season)
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(season_dir, exist_ok=True)

    print(f'\n{"─"*50}')
    print(f'  🌟 SEASON: {season.upper()}  ({len(celebrities)} celebrities)')
    print(f'{"─"*50}')

    for i, query in enumerate(celebrities, 1):
        print(f'  [{i:02}/{len(celebrities)}] {query}')
        download_celebrity(query, raw_dir)

    # Crop faces
    valid_ext = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')
    files = [f for f in os.listdir(raw_dir) if f.lower().endswith(valid_ext)]
    saved = rejected = 0
    for fname in files:
        src = os.path.join(raw_dir, fname)
        dst = os.path.join(season_dir, f'{season}_{fname}')
        if os.path.exists(dst):
            saved += 1; continue
        if crop_face(src, dst): saved += 1
        else:
            shutil.copy2(src, os.path.join(REJECTED_DIR, f'{season}_{fname}'))
            rejected += 1

    print(f'  ✅ Saved: {saved}  ❌ Rejected: {rejected}')
    return saved

# Run collection
to_run = RUN_SEASONS if RUN_SEASONS else list(CELEBRITIES.keys())
print(f'🚀 Running collection for: {to_run}')

for season in to_run:
    collect_season(season, CELEBRITIES[season])

print('\n✅ Data collection complete!')

---
## 🔧 STEP 2 — Preprocessing & Feature Extraction

> Extracts CIELab and HSV color features from each face image.  
> Saves features to a CSV file for traditional ML models.  
> Also prepares the image dataset for MobileNetV2.

In [ ]:
import numpy as np
import pandas as pd
import cv2
import os
from skimage import color
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm

SEASONS      = ['spring', 'summer', 'autumn', 'winter']
FEATURES_CSV = '/content/drive/MyDrive/ToneFit/features.csv'

def extract_features(img_path):
    """
    Extract CIELab + HSV features from a face image.
    Returns a dict of 10 features.
    """
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

    # CIELab conversion
    img_float = img_rgb.astype(np.float32) / 255.0
    img_lab   = color.rgb2lab(img_float)

    L = img_lab[:, :, 0]  # Lightness  (0-100)
    a = img_lab[:, :, 1]  # Red-Green  (-128 to 127)
    b = img_lab[:, :, 2]  # Yellow-Blue(-128 to 127)

    # ITA = Individual Typology Angle
    # Higher ITA = lighter/cooler, Lower ITA = darker/warmer
    L_mean = np.mean(L)
    b_mean = np.mean(b)
    ITA = np.degrees(np.arctan((L_mean - 50) / (b_mean + 1e-6)))

    H = img_hsv[:, :, 0]  # Hue
    S = img_hsv[:, :, 1]  # Saturation
    V = img_hsv[:, :, 2]  # Value

    return {
        'L_mean': L_mean,
        'a_mean': np.mean(a),
        'b_mean': b_mean,
        'L_std':  np.std(L),
        'a_std':  np.std(a),
        'b_std':  np.std(b),
        'ITA':    ITA,
        'H_mean': np.mean(H),
        'S_mean': np.mean(S),
        'V_mean': np.mean(V),
    }

# Build features dataframe
rows = []
valid_ext = ('.jpg', '.jpeg', '.png')

print('🔬 Extracting features...')
for season in SEASONS:
    folder = os.path.join(DATASET_DIR, season)
    if not os.path.exists(folder):
        print(f'  ⚠️ No folder found for {season}, skipping')
        continue

    files = [f for f in os.listdir(folder) if f.lower().endswith(valid_ext)]
    print(f'  Processing {season}: {len(files)} images')

    for fname in tqdm(files, desc=f'  {season}'):
        path = os.path.join(folder, fname)
        feat = extract_features(path)
        if feat is not None:
            feat['filename'] = fname
            feat['season']   = season
            rows.append(feat)

df = pd.DataFrame(rows)
df.to_csv(FEATURES_CSV, index=False)

print(f'\n✅ Features extracted: {len(df)} images')
print(f'   Saved to: {FEATURES_CSV}')
print(f'\n📊 Class distribution:')
print(df['season'].value_counts())
df.head()

In [ ]:
# Prepare train/test splits
FEATURE_COLS = ['L_mean','a_mean','b_mean','L_std','a_std','b_std','ITA','H_mean','S_mean','V_mean']

df = pd.read_csv(FEATURES_CSV)

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['season'])
print(f'Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

X = df[FEATURE_COLS].values
y = df['label'].values

# Normalize features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Stratified train/test split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Save splits
MODELS_DIR = '/content/drive/MyDrive/ToneFit/models'
os.makedirs(MODELS_DIR, exist_ok=True)

np.save(os.path.join(MODELS_DIR, 'X_train.npy'), X_train)
np.save(os.path.join(MODELS_DIR, 'X_test.npy'),  X_test)
np.save(os.path.join(MODELS_DIR, 'y_train.npy'), y_train)
np.save(os.path.join(MODELS_DIR, 'y_test.npy'),  y_test)

# Save scaler and label encoder
import pickle
with open(os.path.join(MODELS_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
with open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

print(f'\n✅ Data splits saved')
print(f'   Train: {len(X_train)} samples')
print(f'   Test : {len(X_test)} samples')

---
## 📊 STEP 3 — Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

df = pd.read_csv(FEATURES_CSV)
SEASON_COLORS = {
    'spring': '#F4A460',
    'summer': '#87CEEB',
    'autumn': '#8B4513',
    'winter': '#4169E1'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('ToneFit ML — Exploratory Data Analysis', fontsize=16, fontweight='bold')

# 1. Class distribution
ax1 = axes[0, 0]
counts = df['season'].value_counts()[['spring','summer','autumn','winter']]
bars = ax1.bar(counts.index, counts.values,
               color=[SEASON_COLORS[s] for s in counts.index], edgecolor='black', linewidth=0.5)
ax1.set_title('Class Distribution', fontweight='bold')
ax1.set_xlabel('Season')
ax1.set_ylabel('Number of Images')
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(val), ha='center', va='bottom', fontweight='bold')

# 2. L* (Lightness) by season — key differentiator
ax2 = axes[0, 1]
for season in ['spring','summer','autumn','winter']:
    subset = df[df['season'] == season]['L_mean']
    ax2.hist(subset, alpha=0.6, label=season.capitalize(),
             color=SEASON_COLORS[season], bins=15, edgecolor='black', linewidth=0.3)
ax2.set_title('L* (Lightness) Distribution by Season', fontweight='bold')
ax2.set_xlabel('L* Mean Value')
ax2.set_ylabel('Frequency')
ax2.legend()

# 3. a* vs b* scatter (undertone visualization)
ax3 = axes[1, 0]
for season in ['spring','summer','autumn','winter']:
    subset = df[df['season'] == season]
    ax3.scatter(subset['a_mean'], subset['b_mean'],
                label=season.capitalize(), color=SEASON_COLORS[season],
                alpha=0.6, s=30, edgecolors='black', linewidths=0.3)
ax3.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)
ax3.axvline(x=0, color='gray', linestyle='--', linewidth=0.8)
ax3.set_title('a* vs b* (Warm/Cool Undertone)', fontweight='bold')
ax3.set_xlabel('a* Mean (Red → Green)')
ax3.set_ylabel('b* Mean (Yellow → Blue)')
ax3.legend()

# 4. ITA Score by season
ax4 = axes[1, 1]
data_by_season = [df[df['season'] == s]['ITA'].values for s in ['spring','summer','autumn','winter']]
bp = ax4.boxplot(data_by_season, labels=['Spring','Summer','Autumn','Winter'],
                 patch_artist=True, notch=False)
for patch, season in zip(bp['boxes'], ['spring','summer','autumn','winter']):
    patch.set_facecolor(SEASON_COLORS[season])
    patch.set_alpha(0.7)
ax4.set_title('ITA Score by Season\n(Higher = Lighter/Cooler)', fontweight='bold')
ax4.set_xlabel('Season')
ax4.set_ylabel('ITA Score')
ax4.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.5)

plt.tight_layout()
RESULTS_DIR = '/content/drive/MyDrive/ToneFit/results'
os.makedirs(RESULTS_DIR, exist_ok=True)
plt.savefig(os.path.join(RESULTS_DIR, 'eda_overview.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved')

In [ ]:
# PCA Visualization — shows if seasons are separable
from sklearn.decomposition import PCA

FEATURE_COLS = ['L_mean','a_mean','b_mean','L_std','a_std','b_std','ITA','H_mean','S_mean','V_mean']
X_pca = MinMaxScaler().fit_transform(df[FEATURE_COLS].values)
pca   = PCA(n_components=2)
X_2d  = pca.fit_transform(X_pca)

plt.figure(figsize=(8, 6))
for season in ['spring','summer','autumn','winter']:
    mask = df['season'].values == season
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                label=season.capitalize(),
                color=SEASON_COLORS[season],
                alpha=0.6, s=40, edgecolors='black', linewidths=0.3)

plt.title(f'PCA — Feature Space\n(Explained variance: {pca.explained_variance_ratio_.sum():.1%})', fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pca_visualization.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ PCA plot saved')

---
## 🤖 STEP 4 — Traditional ML Training
### SVM + Random Forest

> Model A: Uses extracted CIELab + HSV features (10 features per image)  
> Trains SVM with GridSearchCV and Random Forest with feature importance

In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns

# Load data
X_train = np.load(os.path.join(MODELS_DIR, 'X_train.npy'))
X_test  = np.load(os.path.join(MODELS_DIR, 'X_test.npy'))
y_train = np.load(os.path.join(MODELS_DIR, 'y_train.npy'))
y_test  = np.load(os.path.join(MODELS_DIR, 'y_test.npy'))

with open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

season_names = le.classes_.tolist()
season_names_cap = [s.capitalize() for s in season_names]

print(f'✅ Data loaded')
print(f'   Train: {X_train.shape}  Test: {X_test.shape}')
print(f'   Seasons: {season_names}')

In [ ]:
# ── SVM Training ──────────────────────────────────────────────────────────────
print('🔍 Training SVM with GridSearchCV...')
print('   (This may take a few minutes)')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svm_params = {
    'C':     [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01],
    'kernel': ['rbf']
}

svm_grid = GridSearchCV(
    SVC(class_weight='balanced', probability=True, random_state=42),
    svm_params, cv=cv, scoring='f1_weighted', n_jobs=-1, verbose=1
)
svm_grid.fit(X_train, y_train)

best_svm = svm_grid.best_estimator_
print(f'\n✅ Best SVM params: {svm_grid.best_params_}')
print(f'   CV F1 Score: {svm_grid.best_score_:.4f}')

# Save model
with open(os.path.join(MODELS_DIR, 'svm_model.pkl'), 'wb') as f:
    pickle.dump(best_svm, f)
print(f'   Saved to: models/svm_model.pkl')

In [ ]:
# ── Random Forest Training ─────────────────────────────────────────────────────
print('🌲 Training Random Forest...')

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Cross-validation
rf_cv_scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='f1_weighted')
print(f'\n✅ Random Forest trained')
print(f'   CV F1 Score: {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}')

# Feature importance plot
FEATURE_COLS = ['L_mean','a_mean','b_mean','L_std','a_std','b_std','ITA','H_mean','S_mean','V_mean']
importances  = rf.feature_importances_
sorted_idx   = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.bar([FEATURE_COLS[i] for i in sorted_idx], importances[sorted_idx],
        color='steelblue', edgecolor='black', linewidth=0.5)
plt.title('Random Forest — Feature Importance', fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save model
with open(os.path.join(MODELS_DIR, 'rf_model.pkl'), 'wb') as f:
    pickle.dump(rf, f)
print(f'   Saved to: models/rf_model.pkl')

---
## 🧠 STEP 5 — Deep Learning Training
### MobileNetV2 (Transfer Learning)

> Model B: Uses raw 224x224 face images directly  
> Pretrained on ImageNet, fine-tuned for season classification  
> **Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU**

In [ ]:
import tensorflow as tf
print(f'✅ TensorFlow version: {tf.__version__}')
print(f'   GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
if len(tf.config.list_physical_devices('GPU')) > 0:
    print(f'   GPU: {tf.config.list_physical_devices("GPU")[0].name}')
else:
    print('   ⚠️ No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_CLASSES = 4

# ── Data generators ────────────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=42
)

val_gen = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=42
)

print(f'\n✅ Data generators ready')
print(f'   Train batches: {len(train_gen)}')
print(f'   Val batches  : {len(val_gen)}')
print(f'   Classes      : {train_gen.class_indices}')

In [ ]:
# ── Build MobileNetV2 model ────────────────────────────────────────────────────
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base initially

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'✅ MobileNetV2 model built')
print(f'   Total params    : {model.count_params():,}')
print(f'   Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

In [ ]:
# ── Phase 1: Train top layers only ────────────────────────────────────────────
MODEL_PATH = os.path.join(MODELS_DIR, 'mobilenetv2_model.h5')

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor='val_accuracy', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

print('🚀 Phase 1: Training top layers (base frozen)...')
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)
print('\n✅ Phase 1 complete')

In [ ]:
# ── Phase 2: Fine-tune top layers of base model ────────────────────────────────
print('🔧 Phase 2: Fine-tuning top layers of base model...')

# Unfreeze top 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=0.00001),  # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)
print('\n✅ Phase 2 complete')
print(f'   Model saved to: {MODEL_PATH}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MobileNetV2 Training History', fontweight='bold')

# Combine history from both phases
acc  = history1.history['accuracy'] + history2.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
phase_boundary = len(history1.history['accuracy'])

epochs = range(1, len(acc) + 1)

axes[0].plot(epochs, acc, 'b-', label='Train Accuracy')
axes[0].plot(epochs, val_acc, 'r-', label='Val Accuracy')
axes[0].axvline(x=phase_boundary, color='gray', linestyle='--', label='Fine-tune starts')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(epochs, loss, 'b-', label='Train Loss')
axes[1].plot(epochs, val_loss, 'r-', label='Val Loss')
axes[1].axvline(x=phase_boundary, color='gray', linestyle='--', label='Fine-tune starts')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'mobilenetv2_training.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved')

---
## 📈 STEP 6 — Model Evaluation & Comparison

> Evaluates all 3 models on the test set  
> Generates confusion matrices and comparison table

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load models
with open(os.path.join(MODELS_DIR, 'svm_model.pkl'), 'rb') as f:
    svm = pickle.load(f)
with open(os.path.join(MODELS_DIR, 'rf_model.pkl'), 'rb') as f:
    rf = pickle.load(f)
with open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

mobilenet = tf.keras.models.load_model(os.path.join(MODELS_DIR, 'mobilenetv2_model.h5'))

# Load test data
X_test  = np.load(os.path.join(MODELS_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(MODELS_DIR, 'y_test.npy'))

season_names     = le.classes_.tolist()
season_names_cap = [s.capitalize() for s in season_names]

print('✅ All models loaded')

In [ ]:
def evaluate_model(name, y_true, y_pred):
    """Evaluate a model and return metrics dict."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    # Top-2 accuracy — check if correct label is in top 2 predictions
    print(f'\n{"─"*50}')
    print(f'  Model: {name}')
    print(f'{"─"*50}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    print(f'\n  Classification Report:')
    print(classification_report(y_true, y_pred, target_names=season_names_cap))

    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}

def plot_confusion_matrix(name, y_true, y_pred):
    """Plot and save confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=season_names_cap,
                yticklabels=season_names_cap)
    plt.title(f'Confusion Matrix — {name}', fontweight='bold')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(os.path.join(RESULTS_DIR, f'confusion_{safe_name}.png'), dpi=150, bbox_inches='tight')
    plt.show()

results = []

# SVM
svm_pred = svm.predict(X_test)
results.append(evaluate_model('SVM', y_test, svm_pred))
plot_confusion_matrix('SVM', y_test, svm_pred)

# Random Forest
rf_pred = rf.predict(X_test)
results.append(evaluate_model('Random Forest', y_test, rf_pred))
plot_confusion_matrix('Random Forest', y_test, rf_pred)

print('\n⏳ Running MobileNetV2 predictions (may take a moment)...')

In [ ]:
# MobileNetV2 predictions
test_datagen = ImageDataGenerator(rescale=1./255)
test_gen = test_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Map class indices to match label encoder order
class_map  = test_gen.class_indices
y_dl_true  = test_gen.classes
y_dl_pred_probs = mobilenet.predict(test_gen, verbose=1)
y_dl_pred  = np.argmax(y_dl_pred_probs, axis=1)

results.append(evaluate_model('MobileNetV2', y_dl_true, y_dl_pred))
plot_confusion_matrix('MobileNetV2', y_dl_true, y_dl_pred)

In [ ]:
# Final comparison table
df_results = pd.DataFrame(results)
df_results = df_results.set_index('Model')
df_results = df_results.round(4)

print('\n' + '='*55)
print('  📊 FINAL MODEL COMPARISON')
print('='*55)
print(df_results.to_string())
print('='*55)

# Save to CSV
df_results.to_csv(os.path.join(RESULTS_DIR, 'comparison_table.csv'))

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
df_results[['Accuracy','Precision','Recall','F1-Score']].plot(
    kind='bar', ax=ax, edgecolor='black', linewidth=0.5
)
ax.set_title('Model Performance Comparison', fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc='lower right')
ax.axhline(y=0.8, color='red', linestyle='--', alpha=0.5, label='80% threshold')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=8, padding=2)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ All results saved to results/ folder')

---
## 🎨 STEP 7 — Prediction Demo

> Upload a face photo and get your personal color season prediction  
> Shows results from all 3 models + clothing palette

In [ ]:
from google.colab import files
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage import color
import pickle
import tensorflow as tf

# Color palettes per season
PALETTES = {
    'spring': {
        'emoji': '🌸',
        'description': 'Warm · Light · Clear',
        'colors': ['#FFDAB9', '#FF7F50', '#FFD700', '#90EE90', '#87CEEB', '#F4A460'],
        'names':  ['Peach', 'Coral', 'Gold', 'Light Green', 'Sky Blue', 'Sandy Brown'],
        'avoid': 'Avoid: Cool grays, icy blues, stark black'
    },
    'summer': {
        'emoji': '☁️',
        'description': 'Cool · Light · Muted',
        'colors': ['#E6E6FA', '#DDA0DD', '#B0C4DE', '#C1CDC1', '#D8BFD8', '#708090'],
        'names':  ['Lavender', 'Plum', 'Steel Blue', 'Sage', 'Thistle', 'Slate'],
        'avoid': 'Avoid: Warm oranges, mustard yellow, bright neons'
    },
    'autumn': {
        'emoji': '🍂',
        'description': 'Warm · Deep · Muted',
        'colors': ['#8B4513', '#CD853F', '#6B8E23', '#B8860B', '#A0522D', '#556B2F'],
        'names':  ['Saddle Brown', 'Peru', 'Olive', 'Dark Goldenrod', 'Sienna', 'Dark Olive'],
        'avoid': 'Avoid: Cool pastels, icy blues, electric neons'
    },
    'winter': {
        'emoji': '❄️',
        'description': 'Cool · Deep · Clear',
        'colors': ['#000000', '#FFFFFF', '#4169E1', '#DC143C', '#228B22', '#8B008B'],
        'names':  ['Black', 'White', 'Royal Blue', 'Crimson', 'Forest Green', 'Dark Magenta'],
        'avoid': 'Avoid: Warm browns, dusty pastels, muted earth tones'
    }
}

# Load models
with open(os.path.join(MODELS_DIR, 'svm_model.pkl'), 'rb') as f:
    svm = pickle.load(f)
with open(os.path.join(MODELS_DIR, 'rf_model.pkl'), 'rb') as f:
    rf = pickle.load(f)
with open(os.path.join(MODELS_DIR, 'scaler.pkl'), 'rb') as f:
    scaler = pickle.load(f)
with open(os.path.join(MODELS_DIR, 'label_encoder.pkl'), 'rb') as f:
    le = pickle.load(f)

mobilenet = tf.keras.models.load_model(os.path.join(MODELS_DIR, 'mobilenetv2_model.h5'))

CASCADE_PATH = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_cascade = cv2.CascadeClassifier(CASCADE_PATH)

FEATURE_COLS = ['L_mean','a_mean','b_mean','L_std','a_std','b_std','ITA','H_mean','S_mean','V_mean']

def predict_season(img_path):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        print('❌ Could not read image')
        return

    # Detect face
    gray  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))

    if len(faces) == 0:
        print('❌ No face detected. Please use a clear front-facing photo.')
        return

    x, y, w, h = max(faces, key=lambda f: f[2]*f[3])
    pad = int(0.20 * max(w, h))
    x1, y1 = max(0, x-pad), max(0, y-pad)
    x2, y2 = min(img_bgr.shape[1], x+w+pad), min(img_bgr.shape[0], y+h+pad)
    face_crop = cv2.resize(img_bgr[y1:y2, x1:x2], (224, 224))

    # Extract features for traditional ML
    img_rgb   = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
    img_hsv   = cv2.cvtColor(face_crop, cv2.COLOR_BGR2HSV)
    img_lab   = color.rgb2lab(img_rgb.astype(np.float32)/255.0)

    L, a, b = img_lab[:,:,0], img_lab[:,:,1], img_lab[:,:,2]
    L_mean, b_mean = np.mean(L), np.mean(b)
    ITA = np.degrees(np.arctan((L_mean - 50) / (b_mean + 1e-6)))

    feat = np.array([[
        L_mean, np.mean(a), b_mean,
        np.std(L), np.std(a), np.std(b),
        ITA,
        np.mean(img_hsv[:,:,0]),
        np.mean(img_hsv[:,:,1]),
        np.mean(img_hsv[:,:,2])
    ]])
    feat_scaled = scaler.transform(feat)

    # Predictions
    svm_pred    = le.inverse_transform(svm.predict(feat_scaled))[0]
    rf_pred     = le.inverse_transform(rf.predict(feat_scaled))[0]

    img_dl      = img_rgb.astype(np.float32) / 255.0
    img_dl      = np.expand_dims(img_dl, axis=0)
    dl_probs    = mobilenet.predict(img_dl, verbose=0)[0]
    dl_pred_idx = np.argmax(dl_probs)

    # Map DL class index to season name
    # ImageDataGenerator sorts alphabetically: autumn=0, spring=1, summer=2, winter=3
    dl_season_order = sorted(PALETTES.keys())
    dl_pred = dl_season_order[dl_pred_idx]
    dl_conf = dl_probs[dl_pred_idx]

    # Final prediction (majority vote)
    votes = [svm_pred, rf_pred, dl_pred]
    final = max(set(votes), key=votes.count)
    palette = PALETTES[final]

    # Display
    fig = plt.figure(figsize=(16, 8))
    fig.patch.set_facecolor('#1a1a2e')

    # Face image
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.imshow(img_rgb)
    ax1.set_title('Input Face', color='white', fontweight='bold', fontsize=13)
    ax1.axis('off')
    ax1.set_facecolor('#1a1a2e')

    # Predictions panel
    ax2 = fig.add_subplot(1, 3, 2)
    ax2.set_facecolor('#16213e')
    ax2.axis('off')
    ax2.text(0.5, 0.95, f"{palette['emoji']} {final.upper()}",
             transform=ax2.transAxes, ha='center', va='top',
             fontsize=22, fontweight='bold', color='white')
    ax2.text(0.5, 0.82, palette['description'],
             transform=ax2.transAxes, ha='center', va='top',
             fontsize=12, color='#aaaaaa', style='italic')

    model_results = [
        ('SVM', svm_pred),
        ('Random Forest', rf_pred),
        (f'MobileNetV2 ({dl_conf:.0%})', dl_pred)
    ]
    for i, (mname, mpred) in enumerate(model_results):
        y_pos = 0.65 - i * 0.14
        color_dot = '#00ff88' if mpred == final else '#ff6b6b'
        ax2.text(0.15, y_pos, f'● {mname}:', transform=ax2.transAxes,
                 ha='left', va='center', fontsize=11, color=color_dot)
        ax2.text(0.75, y_pos, mpred.capitalize(), transform=ax2.transAxes,
                 ha='center', va='center', fontsize=11,
                 color='white', fontweight='bold')

    ax2.text(0.5, 0.22, palette['avoid'],
             transform=ax2.transAxes, ha='center', va='top',
             fontsize=9, color='#ff9999', wrap=True)

    # Color palette
    ax3 = fig.add_subplot(1, 3, 3)
    ax3.set_facecolor('#16213e')
    ax3.axis('off')
    ax3.text(0.5, 0.97, 'Your Color Palette',
             transform=ax3.transAxes, ha='center', va='top',
             fontsize=13, fontweight='bold', color='white')

    colors  = palette['colors']
    cnames  = palette['names']
    n = len(colors)
    for i, (c, cname) in enumerate(zip(colors, cnames)):
        y_pos = 0.85 - i * (0.78 / n)
        rect = mpatches.FancyBboxPatch(
            (0.05, y_pos - 0.05), 0.25, 0.10,
            boxstyle='round,pad=0.01',
            facecolor=c, edgecolor='white', linewidth=0.5,
            transform=ax3.transAxes
        )
        ax3.add_patch(rect)
        ax3.text(0.36, y_pos, cname, transform=ax3.transAxes,
                 ha='left', va='center', fontsize=10, color='white')
        ax3.text(0.36, y_pos - 0.04, c, transform=ax3.transAxes,
                 ha='left', va='center', fontsize=8, color='#aaaaaa')

    plt.tight_layout(pad=1.5)
    output_path = '/content/tonefit_result.png'
    plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
    plt.show()

    print(f'\n✅ Result saved to: {output_path}')
    print(f'\n  🎨 Final Prediction: {final.upper()}')
    print(f'  SVM          → {svm_pred}')
    print(f'  Random Forest → {rf_pred}')
    print(f'  MobileNetV2  → {dl_pred} ({dl_conf:.1%} confidence)')

print('✅ Prediction function ready!')
print('\nRun the next cell to upload your photo 👇')

In [ ]:
# Upload your photo and get prediction
print('📸 Upload a clear, front-facing face photo:')
uploaded = files.upload()

if uploaded:
    img_path = list(uploaded.keys())[0]
    print(f'\n🔍 Analyzing: {img_path}')
    predict_season(img_path)
else:
    print('No file uploaded.')

---

## ✅ Pipeline Complete!

All outputs are saved to your Google Drive at `/ToneFit/`:

| File | Description |
|------|-------------|
| `dataset/` | Labeled face images per season |
| `features.csv` | Extracted CIELab + HSV features |
| `models/svm_model.pkl` | Trained SVM model |
| `models/rf_model.pkl` | Trained Random Forest model |
| `models/mobilenetv2_model.h5` | Trained MobileNetV2 model |
| `models/scaler.pkl` | Feature scaler |
| `models/label_encoder.pkl` | Label encoder |
| `results/comparison_table.csv` | Final model comparison |
| `results/confusion_*.png` | Confusion matrices |
| `results/model_comparison.png` | Performance bar chart |
| `results/eda_overview.png` | EDA visualizations |

---

**Next steps for your paper:**
1. Fill in the TBD values in your comparison table from `results/comparison_table.csv`
2. Use the confusion matrices to discuss which seasons were hardest to classify
3. Note whether Autumn was the most misclassified (expected finding)
4. Compare your accuracy against DeepColor's 81.12% baseline